# Convolutional Autoencoder on SVHN Dataset

This notebook implements a deep convolutional autoencoder for unsupervised representation learning on the Street View House Numbers (SVHN) dataset.

**Objective**: Learn compressed latent representations of images through reconstruction.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Load SVHN dataset using TensorFlow Datasets
import tensorflow_datasets as tfds

print("Loading SVHN dataset...")

# Load train and test splits
ds_train = tfds.load('svhn_cropped', split='train', as_supervised=False)
ds_test = tfds.load('svhn_cropped', split='test', as_supervised=False)

# Convert to numpy arrays
def dataset_to_numpy(dataset, max_samples=None):
    images = []
    for i, example in enumerate(dataset):
        if max_samples and i >= max_samples:
            break
        images.append(example['image'].numpy())
    return np.array(images)

# Load subsets
print("Extracting training data...")
X_train = dataset_to_numpy(ds_train, max_samples=30000)
print("Extracting validation data...")
X_val = dataset_to_numpy(ds_test, max_samples=5000)

# Normalize to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Image value range: [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.axis('off')
plt.suptitle('Sample SVHN Images (32x32 RGB)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Model Architecture

We implement a deep convolutional autoencoder with the following structure:

**Encoder**: Progressive downsampling using convolution layers with stride 2
- Input: 32×32×3
- Conv layers reduce spatial dimensions while increasing channels
- Output: Flattened latent vector of specified dimension

**Decoder**: Progressive upsampling to reconstruct the original image
- Input: Latent vector
- Transpose convolutions restore spatial dimensions
- Output: 32×32×3 reconstructed image

In [ ]:
def build_encoder(latent_dim):
    """Build encoder: compresses 32x32x3 image to latent_dim vector"""
    encoder_input = layers.Input(shape=(32, 32, 3))
    
    # Convolutional layers with progressive downsampling
    x = layers.Conv2D(32, 3, strides=2, padding='same')(encoder_input)  # 16x16x32
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)  # 8x8x64
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)  # 4x4x128
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)  # 2x2x256
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Flatten and compress to latent dimension
    x = layers.Flatten()(x)
    latent = layers.Dense(latent_dim, name='latent')(x)
    
    return models.Model(encoder_input, latent, name='encoder')


def build_decoder(latent_dim):
    """Build decoder: reconstructs 32x32x3 image from latent_dim vector"""
    decoder_input = layers.Input(shape=(latent_dim,))
    
    # Expand latent vector back to spatial dimensions
    x = layers.Dense(2 * 2 * 256)(decoder_input)
    x = layers.Reshape((2, 2, 256))(x)
    
    # Transpose convolutions for upsampling
    x = layers.Conv2DTranspose(128, 3, strides=2, padding='same')(x)  # 4x4x128
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2DTranspose(64, 3, strides=2, padding='same')(x)  # 8x8x64
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.Conv2DTranspose(32, 3, strides=2, padding='same')(x)  # 16x16x32
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Final layer: reconstruct RGB image
    decoder_output = layers.Conv2DTranspose(3, 3, strides=2, padding='same', activation='sigmoid')(x)  # 32x32x3
    
    return models.Model(decoder_input, decoder_output, name='decoder')


def build_autoencoder(latent_dim):
    """Combine encoder and decoder into full autoencoder"""
    encoder = build_encoder(latent_dim)
    decoder = build_decoder(latent_dim)
    
    autoencoder_input = layers.Input(shape=(32, 32, 3))
    encoded = encoder(autoencoder_input)
    decoded = decoder(encoded)
    
    autoencoder = models.Model(autoencoder_input, decoded, name='autoencoder')
    
    return autoencoder, encoder, decoder

## 3. Training Configuration

In [ ]:
# Training parameters
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.001

## 4. Experiment 1: Latent Dimension = 64

First, we train an autoencoder with a latent dimension of 64. This provides a good balance between compression and reconstruction quality.

In [ ]:
# Build model with latent_dim=64
autoencoder_64, encoder_64, decoder_64 = build_autoencoder(latent_dim=64)

autoencoder_64.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

print("\n=== Autoencoder with Latent Dimension 64 ===")
autoencoder_64.summary()

In [ ]:
# Learning rate scheduler for better convergence
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train the model
print("\nTraining autoencoder with latent dimension 64...")
history_64 = autoencoder_64.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, X_val),
    callbacks=[lr_scheduler],
    verbose=1
)

print("\nTraining completed!")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history_64.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history_64.history['val_loss'], label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Loss Curve - Latent Dim 64')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_64.history['mae'], label='Training MAE', linewidth=2)
ax2.plot(history_64.history['val_mae'], label='Validation MAE', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.set_title('Mean Absolute Error - Latent Dim 64')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_train_loss = history_64.history['loss'][-1]
final_val_loss = history_64.history['val_loss'][-1]
print(f"Final Training Loss: {final_train_loss:.6f}")
print(f"Final Validation Loss: {final_val_loss:.6f}")

### 4.1 Reconstruction Results (Latent Dim 64)

In [ ]:
# Generate reconstructions
n_samples = 10
sample_images = X_val[:n_samples]
reconstructed_64 = autoencoder_64.predict(sample_images, verbose=0)

# Display original vs reconstructed
fig, axes = plt.subplots(2, n_samples, figsize=(20, 4))

for i in range(n_samples):
    # Original images
    axes[0, i].imshow(sample_images[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=12, fontweight='bold')
    
    # Reconstructed images
    axes[1, i].imshow(reconstructed_64[i])
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=12, fontweight='bold')

plt.suptitle('Image Reconstruction - Latent Dimension 64', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate reconstruction error
mse = np.mean((sample_images - reconstructed_64) ** 2)
print(f"Average reconstruction MSE: {mse:.6f}")

### 4.2 Latent Representation Analysis (Latent Dim 64)

In [ ]:
# Extract latent vectors
latent_vectors_64 = encoder_64.predict(sample_images, verbose=0)

print(f"Latent representation shape: {latent_vectors_64.shape}")
print(f"Each image is compressed to a vector of length {latent_vectors_64.shape[1]}")
print(f"\nCompression ratio: {(32*32*3) / latent_vectors_64.shape[1]:.2f}x")
print(f"\nExample latent vector (first image):")
print(latent_vectors_64[0])
print(f"\nLatent vector statistics:")
print(f"  Mean: {np.mean(latent_vectors_64[0]):.4f}")
print(f"  Std: {np.std(latent_vectors_64[0]):.4f}")
print(f"  Min: {np.min(latent_vectors_64[0]):.4f}")
print(f"  Max: {np.max(latent_vectors_64[0]):.4f}")

## 5. Experiment 2: Latent Dimension = 16

Now we train a second autoencoder with a much smaller latent dimension of 16. This creates a higher compression ratio and tests how much information can be preserved in a very compact representation.

In [ ]:
# Build model with latent_dim=16
autoencoder_16, encoder_16, decoder_16 = build_autoencoder(latent_dim=16)

autoencoder_16.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

print("\n=== Autoencoder with Latent Dimension 16 ===")
autoencoder_16.summary()

In [ ]:
# Train the model
lr_scheduler_16 = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

print("\nTraining autoencoder with latent dimension 16...")
history_16 = autoencoder_16.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, X_val),
    callbacks=[lr_scheduler_16],
    verbose=1
)

print("\nTraining completed!")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history_16.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history_16.history['val_loss'], label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Loss Curve - Latent Dim 16')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_16.history['mae'], label='Training MAE', linewidth=2)
ax2.plot(history_16.history['val_mae'], label='Validation MAE', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.set_title('Mean Absolute Error - Latent Dim 16')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_train_loss_16 = history_16.history['loss'][-1]
final_val_loss_16 = history_16.history['val_loss'][-1]
print(f"Final Training Loss: {final_train_loss_16:.6f}")
print(f"Final Validation Loss: {final_val_loss_16:.6f}")

### 5.1 Reconstruction Results (Latent Dim 16)

In [ ]:
# Generate reconstructions
reconstructed_16 = autoencoder_16.predict(sample_images, verbose=0)

# Display original vs reconstructed
fig, axes = plt.subplots(2, n_samples, figsize=(20, 4))

for i in range(n_samples):
    # Original images
    axes[0, i].imshow(sample_images[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=12, fontweight='bold')
    
    # Reconstructed images
    axes[1, i].imshow(reconstructed_16[i])
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=12, fontweight='bold')

plt.suptitle('Image Reconstruction - Latent Dimension 16', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate reconstruction error
mse_16 = np.mean((sample_images - reconstructed_16) ** 2)
print(f"Average reconstruction MSE: {mse_16:.6f}")

### 5.2 Latent Representation Analysis (Latent Dim 16)

In [ ]:
# Extract latent vectors
latent_vectors_16 = encoder_16.predict(sample_images, verbose=0)

print(f"Latent representation shape: {latent_vectors_16.shape}")
print(f"Each image is compressed to a vector of length {latent_vectors_16.shape[1]}")
print(f"\nCompression ratio: {(32*32*3) / latent_vectors_16.shape[1]:.2f}x")
print(f"\nExample latent vector (first image):")
print(latent_vectors_16[0])
print(f"\nLatent vector statistics:")
print(f"  Mean: {np.mean(latent_vectors_16[0]):.4f}")
print(f"  Std: {np.std(latent_vectors_16[0]):.4f}")
print(f"  Min: {np.min(latent_vectors_16[0]):.4f}")
print(f"  Max: {np.max(latent_vectors_16[0]):.4f}")

## 6. Comparison: Latent Dimension 64 vs 16

Here we directly compare the reconstruction quality of both models side by side.

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(3, n_samples, figsize=(20, 6))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(sample_images[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=12, fontweight='bold')
    
    # Latent dim 64
    axes[1, i].imshow(reconstructed_64[i])
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Latent=64', fontsize=12, fontweight='bold')
    
    # Latent dim 16
    axes[2, i].imshow(reconstructed_16[i])
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title('Latent=16', fontsize=12, fontweight='bold')

plt.suptitle('Reconstruction Quality Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison
print("=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"\nLatent Dimension 64:")
print(f"  Compression ratio: {(32*32*3)/64:.2f}x")
print(f"  Final validation loss: {final_val_loss:.6f}")
print(f"  Reconstruction MSE: {mse:.6f}")

print(f"\nLatent Dimension 16:")
print(f"  Compression ratio: {(32*32*3)/16:.2f}x")
print(f"  Final validation loss: {final_val_loss_16:.6f}")
print(f"  Reconstruction MSE: {mse_16:.6f}")

print(f"\nQuality Difference:")
loss_increase = ((final_val_loss_16 - final_val_loss) / final_val_loss) * 100
print(f"  Loss increase with 4x more compression: {loss_increase:.2f}%")
print("=" * 60)

## 7. Understanding Autoencoders and Latent Representations

### What is a Latent Representation?

A **latent representation** is a compressed, encoded version of the input data that captures its essential features. Think of it as a "summary" or "fingerprint" of the image. The encoder network learns to extract the most important information from the 32×32×3 image (3,072 values) and compress it into a much smaller vector (64 or 16 values).

This compressed representation lives in the "latent space" - a lower-dimensional space where similar images have similar representations. The network learns this encoding in an unsupervised way by trying to reconstruct the original image.

### How Dimensionality Reduction Affects Reconstruction

The **latent dimension** determines how much information can be preserved:

- **Higher dimension (64)**: More capacity to store details. The model can preserve finer features like exact colors, edges, and textures. Results in better reconstruction quality but less compression.

- **Lower dimension (16)**: Forces the model to be more selective about what to preserve. It learns to capture only the most essential features. This results in higher compression (192x vs 48x) but some loss of fine details.

From our experiments, reducing the latent dimension from 64 to 16 (4x reduction) increased the reconstruction error, demonstrating the fundamental trade-off between compression and reconstruction quality.

### The Role of Autoencoders in Representation Learning

Autoencoders serve several important purposes in machine learning:

1. **Dimensionality Reduction**: They learn to compress high-dimensional data into compact representations, similar to PCA but non-linear and more powerful.

2. **Feature Learning**: The encoder learns useful features automatically without labels. These learned features can be transferred to other tasks (transfer learning).

3. **Denoising**: Autoencoders can be trained to remove noise from data by learning to reconstruct clean versions of corrupted inputs.

4. **Anomaly Detection**: Images that reconstruct poorly likely differ from the training distribution, useful for detecting outliers.

5. **Data Generation**: The decoder can generate new images by sampling from the latent space, though VAEs (Variational Autoencoders) are better suited for this.

In our case, the autoencoder successfully learned to compress SVHN digit images into compact representations while preserving enough information to reconstruct recognizable digits, demonstrating that meaningful structure exists in the learned latent space.

## 8. Conclusion

This notebook demonstrated:

✓ Implementation of a deep convolutional autoencoder on SVHN dataset  
✓ Training with 30,000 images for 50 epochs  
✓ Comparison of two latent dimensions (64 and 16)  
✓ Successful image reconstruction with MSE loss  
✓ Analysis of latent representations and compression ratios  
✓ Visualization of reconstruction quality  

The results show that even with high compression ratios (up to 192x), the autoencoder preserves sufficient information to reconstruct recognizable digit images, validating the effectiveness of learned latent representations.